In [ ]:
"""
============================================================
  Monte Carlo Estimation of the Volume of the n-Ball
  Core function: nSphereVolume(dim, iterations)
============================================================

Theory
------
The volume of the unit n-ball is:
    V_n = pi^(n/2) / Gamma(n/2 + 1)

For n=10:
    V_10 = pi^5 / 120  ≈  2.5501640399

Monte Carlo principle:
    Sample points uniformly in the hypercube [-1, 1]^n  (vol = 2^n).
    A point x is inside the unit ball if ||x|| < 1.
    V_n  ≈  (hits / total) * 2^n
"""

import numpy as np
from scipy.special import gamma
import time

def VolumeSphere(dim, iterations):
    """
    Estimate the volume of the unit n-ball using hit-or-miss Monte Carlo.

    Parameters
    ----------
    dim        : int   – number of dimensions
    iterations : int   – number of random sample points

    Returns
    -------
    float  – estimated volume  ≈  V_n(r=1)
    """
    count_in_sphere = 0

    for count_loops in range(iterations):
        point    = np.random.uniform(-1.0, 1.0, dim)
        distance = np.linalg.norm(point)
        if distance < 1.0:
            count_in_sphere += 1

    return np.power(2.0, dim) * (count_in_sphere / iterations)


def exact_ball_volume(dim):
    return (np.pi ** (dim / 2.0)) / gamma(dim / 2.0 + 1.0)

dimension = 10
ITERATIONS = 1000000
EXACT      = exact_ball_volume(dimension)

np.random.seed(42)

print(f"\n{'='*56}")
print(f"  Monte Carlo — Volume of the {dimension}-Ball  (r=1)")
print(f"{'='*56}")
print(f"  Exact value  π⁵/120  =  {EXACT:.10f}")
print(f"{'─'*56}")

t0       = time.time()
estimate = VolumeSphere(dimension, ITERATIONS)
elapsed  = time.time() - t0

abs_err = abs(estimate - EXACT)
rel_err = 100.0 * abs_err / EXACT

print(f"  Iterations              : {ITERATIONS:,}")
print(f"  Estimated volume        : {estimate:.8f}")
print(f"  Exact volume            : {EXACT:.8f}")
print(f"  Absolute error          : {abs_err:.8f}")
print(f"  Relative error          : {rel_err:.4f} %")
print(f"  Run time                : {elapsed:.2f} s")
print(f"{'='*56}\n")

checkpoints = [1000, 5000, 10000, 50000, 100000, 500000, 1000000]

print(f"{'─'*56}")
print(f"  Convergence study  (dim={dimension})")
print(f"{'─'*56}")
print(f"  {'Iterations':>12}  {'Estimate':>10}  {'Abs Err':>9}  {'Rel Err %':>10}  {'Time (s)':>8}")
print(f"  {'─'*12}  {'─'*10}  {'─'*9}  {'─'*10}  {'─'*8}")

np.random.seed(42)
for n in checkpoints:
    t0  = time.time()
    est = VolumeSphere(dimension, n)
    dt  = time.time() - t0
    ae  = abs(est - EXACT)
    re  = 100.0 * ae / EXACT
    print(f"  {n:>12,}  {est:>10.6f}  {ae:>9.6f}  {re:>10.4f}  {dt:>8.2f}")

print()


ITER_SWEEP = 200000   # keep runtime reasonable across many dims

print(f"{'─'*56}")
print(f"  Dimension sweep  ({ITER_SWEEP:,} iterations per dim)")
print(f"{'─'*56}")
print(f"  {'dim':>4}  {'Exact V_n':>12}  {'MC Estimate':>12}  {'Rel Err %':>10}")
print(f"  {'─'*4}  {'─'*12}  {'─'*12}  {'─'*10}")

np.random.seed(42)
for d in range(1, 13):
    exact_d = exact_ball_volume(d)
    est_d   = VolumeSphere(d, ITER_SWEEP)
    re_d    = 100.0 * abs(est_d - exact_d) / exact_d
    marker  = "  ← n=10" if d == 10 else ""
    print(f"  {d:>4}  {exact_d:>12.6f}  {est_d:>12.6f}  {re_d:>10.4f}{marker}")

print()

p_exact  = EXACT / np.power(2.0, dimension)          # hit probability ≈ 0.00249
theo_std = np.power(2.0, dimension) * np.sqrt(p_exact * (1 - p_exact) / ITERATIONS)

print(f"{'─'*56}")
print(f"  Estimator statistics  (n={dimension},  N={ITERATIONS:,})")
print(f"{'─'*56}")
print(f"  Hit probability p       = {p_exact:.6e}")
print(f"  Theoretical Std[V̂_N]   = {theo_std:.6f}")
print(f"  95%% CI half-width       = {1.96*theo_std:.6f}")
print(f"  Observed absolute error = {abs_err:.6f}")
print(f"  Obs. error / Theo. Std  = {abs_err/theo_std:.2f}σ")
print(f"{'='*56}\n")


  Monte Carlo — Volume of the 10-Ball  (r=1)
  Exact value  π⁵/120  =  2.5501640399
────────────────────────────────────────────────────────
  Iterations              : 1,000,000
  Estimated volume        : 2.58150400
  Exact volume            : 2.55016404
  Absolute error          : 0.03133996
  Relative error          : 1.2289 %
  Run time                : 6.03 s

────────────────────────────────────────────────────────
  Convergence study  (dim=10)
────────────────────────────────────────────────────────
    Iterations    Estimate    Abs Err   Rel Err %  Time (s)
  ────────────  ──────────  ─────────  ──────────  ────────
         1,000    1.024000   1.526164     59.8457      0.01
         5,000    2.457600   0.092564      3.6297      0.03
        10,000    2.867200   0.317036     12.4320      0.06
        50,000    2.457600   0.092564      3.6297      0.31
       100,000    2.672640   0.122476      4.8027      0.62
       500,000    2.519040   0.031124      1.2205      3.02
     1